# 02 - Compare experiments
Reads `outputs/experiments.csv` (one row per training run) and makes the comparison tables and charts for the report: backbone comparison, preprocessing ablation, balancing comparison and hyperparameter tuning.

In [ ]:
# Run from the notebooks/ folder: move to the project root so imports and paths work
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import config
config.ensure_dirs()
config.set_seed()

In [ ]:
log = pd.read_csv(config.EXPERIMENT_LOG)
log = log[log["exp_name"] != "quick_test"].drop_duplicates("exp_name", keep="last")
cols = ["exp_name", "backbone", "preprocess", "balance", "dropout", "lr_finetune", "unfreeze",
        "epochs_run", "train_minutes", "params", "val_qwk", "test_accuracy", "test_macro_f1",
        "test_qwk", "test_dr_sensitivity", "test_dr_specificity", "test_dr_auc"]
log[cols].sort_values("test_qwk", ascending=False).round(3)

## Chart helper

In [ ]:
def compare(rows, label_col, title, fname, metrics=("test_accuracy", "test_macro_f1", "test_qwk")):
    """Grouped bar chart of the chosen metrics for a subset of experiments."""
    data = rows.set_index(label_col)[list(metrics)]
    ax = data.plot(kind="bar", figsize=(8, 4), rot=0, ylim=(0, 1))
    for c in ax.containers:
        ax.bar_label(c, fmt="%.2f", fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("")
    plt.tight_layout()
    plt.savefig(config.FIGURES_DIR / fname, dpi=150, bbox_inches="tight")
    plt.show()
    return data.round(3)

## Backbone comparison (same preprocessing and balancing)

In [ ]:
rows = log[(log["preprocess"] == "full") & (log["balance"] == "weights")].drop_duplicates("backbone", keep="last")
compare(rows, "backbone", "Backbone comparison (test set)", "compare_backbones.png")

## Preprocessing ablation (same backbone)

In [ ]:
best_backbone = log.sort_values("test_qwk", ascending=False).iloc[0]["backbone"]
rows = log[(log["backbone"] == best_backbone) & (log["balance"] == "weights")].drop_duplicates("preprocess", keep="last")
compare(rows, "preprocess", f"Preprocessing ablation ({best_backbone}, validation set)", "compare_preprocessing.png",
        metrics=("val_accuracy", "val_macro_f1", "val_qwk"))

Use **validation** scores for design decisions (ablation, tuning) and report **test** scores only for the final model; choosing settings by test score would leak information from the test set.

## Class-balancing comparison

In [ ]:
rows = log[(log["backbone"] == best_backbone) & (log["preprocess"] == "full")].drop_duplicates("balance", keep="last")
compare(rows, "balance", "Balancing method comparison (validation set)", "compare_balancing.png",
        metrics=("val_accuracy", "val_macro_f1", "val_qwk"))

## Per-class recall of the best run (shows whether rare stages improved)

In [ ]:
best = log.sort_values("val_qwk", ascending=False).iloc[0]["exp_name"]
print("Best run by validation QWK:", best)
rep = pd.read_csv(config.MODELS_DIR / best / "test" / "classification_report.csv", index_col=0)
rep.round(3)